In [0]:
%sql
MERGE INTO retail_lakehouse.gold.dim_customer tgt
USING retail_lakehouse.silver.customers src
ON tgt.CustomerID = src.CustomerID
AND tgt.IsActive = TRUE
WHEN MATCHED
AND (
    tgt.City <> src.City
    OR tgt.Address <> src.Address
    OR tgt.Email <> src.Email
)
THEN UPDATE SET
    tgt.EndDate = CURRENT_DATE(),
    tgt.IsActive = FALSE;INSERT INTO retail_lakehouse.gold.dim_customer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)

SELECT
    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,
    CURRENT_DATE(),
    DATE '9999-12-31',
    TRUE
FROM retail_lakehouse.silver.customers src
LEFT JOIN retail_lakehouse.gold.dim_customer tgt
ON src.CustomerID = tgt.CustomerID
AND tgt.IsActive = TRUE
WHERE tgt.CustomerID IS NULL;